# S3: ограниченная работа с объектами


Показать список, пагинацию, метаданные объекта и сравнение с файлами Iceberg.

Нужны `boto3` и переменные `MINIO_ENDPOINT`, `MINIO_ACCESS_KEY`, `MINIO_SECRET_KEY`; отдельные примеры также используют `EXAMPLE_BUCKET` и `EXAMPLE_OBJECT_KEY`. Без настроек S3 соответствующие ячейки сообщают о пропуске.

Ограничения: Для диагностики хранилища, проверки наличия объекта и понимания границы между S3 и Iceberg. Не печатайте ключи; список объектов не является списком строк или согласованной версией таблицы.


## Основные понятия
S3 хранит объекты, а Iceberg определяет состав версии таблицы. Клиент создаётся только при наличии переменных окружения.


In [ ]:
from lakehouse_spark import SparkSessionContext
from pyspark.sql import functions as F

import os
import boto3

## Примеры


**1. Конфигурация**

Проверьте наличие настройки без вывода ключей.



In [ ]:
endpoint = os.getenv("MINIO_ENDPOINT")
access = os.getenv("MINIO_ACCESS_KEY")
secret = os.getenv("MINIO_SECRET_KEY")

print("S3 настроен:", bool(endpoint and access and secret))

**2. Ограниченный список**

Получите не более двадцати объектов из заданного контейнера.



In [ ]:
if not os.getenv("MINIO_ACCESS_KEY") or not os.getenv("MINIO_SECRET_KEY"):
    print("Пример пропущен: ключи S3 не заданы")
else:
    client = boto3.client(
        "s3", 
        endpoint_url=os.getenv("MINIO_ENDPOINT", "http://minio:9000"), 
        aws_access_key_id=os.environ["MINIO_ACCESS_KEY"], 
        aws_secret_access_key=os.environ["MINIO_SECRET_KEY"]
    )
    page = client.list_objects_v2(Bucket=os.getenv("EXAMPLE_BUCKET", "warehouse"), MaxKeys=20)
    
    print(*[(item["Key"], item["Size"]) for item in page.get("Contents", [])], sep="\n")

**3. Постраничное чтение**

Остановите обход после двух страниц, даже если объектов больше.



In [ ]:
if not os.getenv("MINIO_ACCESS_KEY") or not os.getenv("MINIO_SECRET_KEY"):
    print("Пример пропущен: ключи S3 не заданы")
else:
    client = boto3.client(
        "s3", 
        endpoint_url=os.getenv("MINIO_ENDPOINT", "http://minio:9000"), 
        aws_access_key_id=os.environ["MINIO_ACCESS_KEY"], 
        aws_secret_access_key=os.environ["MINIO_SECRET_KEY"]
    )
    paginator = client.get_paginator("list_objects_v2")
    for number, page in enumerate(
        paginator.paginate(Bucket=os.getenv("EXAMPLE_BUCKET", "warehouse"), 
        PaginationConfig={"MaxItems": 40, "PageSize": 20}), 
        1
    ):
        print("Страница:", number, "объектов:", len(page.get("Contents", [])))
        if number == 2:
            break


**4. Метаданные объекта**

Запросите размер и дату одного известного ключа из окружения.



In [ ]:
if not os.getenv("MINIO_ACCESS_KEY") or not os.getenv("MINIO_SECRET_KEY") or not os.getenv("EXAMPLE_OBJECT_KEY"):
    print("Пример пропущен: настройки объекта не заданы")
else:
    client = boto3.client(
        "s3", 
        endpoint_url=os.getenv("MINIO_ENDPOINT", "http://minio:9000"), 
        aws_access_key_id=os.environ["MINIO_ACCESS_KEY"], 
        aws_secret_access_key=os.environ["MINIO_SECRET_KEY"]
    )
    info = client.head_object(Bucket=os.getenv("EXAMPLE_BUCKET", "warehouse"), Key=os.environ["EXAMPLE_OBJECT_KEY"])
    
    print({"size": info.get("ContentLength"), "modified": info.get("LastModified"), "type": info.get("ContentType")})


**5. Файлы Iceberg**

Сопоставьте список файлов таблицы с метаданными Iceberg, не обходя хранилище вручную.



In [ ]:
with SparkSessionContext('example-s3-iceberg') as spark:
    spark.sql("""
        select file_path, record_count, file_size_in_bytes 
        from lakehouse.raw.pg__matches.files 
        order by file_size_in_bytes desc
    """).show(20, truncate=False)
